# EfficientNet-B0, 2.5D input, soft voting, ensemble — and a leakage demonstration

Closes out Phase D. Four things happen here:

1. **EfficientNet-B0 from scratch** — the third architecture, same protocol as the others.
2. **2.5D input** — stack three *adjacent* axial slices as the three channels instead of
   copying one slice three times. Same tensor shape, same parameter count, real
   volumetric context.
3. **Soft voting and a 3-model ensemble** — cheap aggregation wins, no new training.
4. **A deliberately leaky split** — reproducing the standard methodological bug in this
   literature to measure exactly how many points it invents.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import CLASSES, build_dataloaders, build_dataloaders_25d, compute_class_weights
from models import SimpleCNN, build_mobilenetv2, build_efficientnet_b0
from train import train_model
from evaluate import (get_predictions, slice_level_report, subject_level_report,
                      subject_level_soft_vote, ensemble_predictions)

%matplotlib inline
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '-', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU only')

manifest = pd.read_csv('../data/manifest.csv')
class_weights = compute_class_weights(manifest)
torch.manual_seed(42); np.random.seed(42)
print(manifest.groupby(['class', 'split']).size().unstack())

In [ ]:
def plot_cm(cm, title, ax=None):
    """Confusion matrix with per-class recall on the diagonal made obvious --
    aggregate accuracy hides which class the model is actually failing on."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title(title)
    return ax

## 1. EfficientNet-B0 from scratch

`pretrained=False` by default now, so finding 7 can't be repeated by accident.
At 4.0M parameters this is the largest model tried on ~10.8k training images, so
overfitting is the thing to watch in the curves below.

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(
    manifest, batch_size=32, num_workers=2, rgb=True)

effnet = build_efficientnet_b0(num_classes=len(CLASSES), pretrained=False)
print(f'EfficientNet-B0: {sum(p.numel() for p in effnet.parameters()):,} parameters')

history = train_model(effnet, train_loader, val_loader, class_weights, device,
                      epochs=40, lr=1e-3, patience=7, weight_decay=1e-4,
                      checkpoint_path='../models/checkpoints/efficientnet_b0_scratch.pt')

In [ ]:
preds_effnet = get_predictions(effnet, test_loader, device)
print('===== SUBJECT LEVEL — soft vote =====')
cm_effnet, subj_effnet = subject_level_soft_vote(preds_effnet)
plot_cm(cm_effnet, 'EfficientNet-B0, subject soft vote')

## 2. 2.5D input — three adjacent slices instead of three copies

The standard grayscale→fake-RGB trick feeds the network **three identical copies** of one
slice, so two thirds of the input carries no information at all. `MRI25DDataset` stacks
slices *i-1, i, i+1* instead.

Why this should help on this task specifically: a dark region on a single 2D slice can be
noise or a partial-volume artifact, whereas genuine hippocampal atrophy persists across
consecutive slices. A single-slice model cannot distinguish those two cases even in
principle. This one can, at zero extra parameter cost.

Both augmentation and normalization are handled carefully — the same geometric transform
is applied to all three channels at once (augmenting them independently would misalign the
anatomy across depth and destroy the relationship being exploited), and `T.Grayscale(3)`
is deliberately absent from the 2.5D transform since it would collapse the stack back into
three identical channels.

In [ ]:
tr25, va25, te25 = build_dataloaders_25d(manifest, batch_size=32, num_workers=2)

# proof the channels really are different slices, not three copies
imgs, labels, sids = next(iter(te25))
img = imgs[0]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for i, name in enumerate(['slice i-1', 'slice i', 'slice i+1']):
    axes[i].imshow(img[i], cmap='gray'); axes[i].set_title(name); axes[i].axis('off')
axes[3].imshow((img[2] - img[0]).abs(), cmap='magma')
axes[3].set_title('|i+1 - i-1| (nonzero = real depth)'); axes[3].axis('off')
plt.tight_layout()
print('mean abs difference between adjacent channels:', (img[0]-img[1]).abs().mean().item())

In [ ]:
effnet25 = build_efficientnet_b0(num_classes=len(CLASSES), pretrained=False)
hist25 = train_model(effnet25, tr25, va25, class_weights, device,
                     epochs=40, lr=1e-3, patience=7, weight_decay=1e-4,
                     checkpoint_path='../models/checkpoints/efficientnet_b0_honest25d.pt')

preds_25d = get_predictions(effnet25, te25, device)
cm_25d, subj_25d = subject_level_soft_vote(preds_25d)
plot_cm(cm_25d, 'EfficientNet-B0 2.5D, subject soft vote')

## 3. Hard vote vs soft vote

Majority voting discards confidence. If 17 slices weakly prefer EMCI (p=0.30 each) and 15
slices strongly say AD (p=0.85 each), majority voting returns EMCI even though the averaged
evidence clearly points to AD. Soft voting keeps that magnitude information, which matters
most when individual slices are near chance — which is the regime this dataset is in.

In [ ]:
for name, p in [('EfficientNet-B0', preds_effnet), ('EfficientNet-B0 2.5D', preds_25d)]:
    _, sh = subject_level_report(p)
    _, ss = subject_level_soft_vote(p, verbose=False)
    hard = (sh['true'] == sh['pred']).mean()
    soft = (ss['true'] == ss['pred']).mean()
    print(f'{name:24s} hard {hard:.3f} | soft {soft:.3f} | delta {soft-hard:+.3f}')

## 4. Three-model ensemble

Averaging softmax across the custom CNN, MobileNetV2 and EfficientNet-B0. The models fail
on *different* subjects, so averaging lets a confidently-correct model outvote two
unconfidently-wrong ones. All prediction frames come from loaders built with
`shuffle=False`, so their rows correspond to the same images — `ensemble_predictions`
asserts this rather than trusting it.

In [ ]:
cnn = SimpleCNN(num_classes=4, in_channels=1)
cnn.load_state_dict(torch.load('../models/checkpoints/custom_cnn.pt')); cnn.to(device).eval()
_, _, test_gray = build_dataloaders(manifest, batch_size=32, num_workers=2, rgb=False)
preds_cnn = get_predictions(cnn, test_gray, device)

mnet = build_mobilenetv2(4, pretrained=False)
mnet.load_state_dict(torch.load('../models/checkpoints/mobilenetv2_honest2d.pt')); mnet.to(device).eval()
preds_mnet = get_predictions(mnet, test_loader, device)

ens = ensemble_predictions([preds_cnn, preds_mnet, preds_effnet])
print('===== ENSEMBLE — subject level, soft vote =====')
cm_ens, subj_ens = subject_level_soft_vote(ens)
plot_cm(cm_ens, 'ensemble, subject soft vote')

## 5. Leakage demonstration — why this project splits by subject

Every result above splits by **subject**: a person's 32 slices are entirely in train, or
entirely in test, never both. The common alternative in published work is to shuffle
*slices*. Below, the same models are retrained on a slice-wise split and nothing else is
changed.

The accuracy will jump enormously. It is not a better model. Each subject contributes 32
near-duplicate axial slices, so slice-wise splitting puts the same brain on both sides of
the boundary and the network scores well by **recognizing the individual**, not by
detecting atrophy — it would score similarly if the diagnoses were shuffled at random.

Roughly half the published Alzheimer's-MRI deep learning literature reports 90%+ accuracy
obtained this way. Studies that split properly by subject report AD-vs-CN *binary*
accuracy under 71% on ADNI-sized data. That is the gap this cell measures.

In [ ]:
leaky_results = {}
for arch in ['custom_cnn', 'mobilenetv2', 'efficientnet_b0']:
    path = f'../reports/{arch}_leaky_result.json'
    try:
        leaky_results[arch] = json.load(open(path))
    except FileNotFoundError:
        print('not yet run:', path)

honest = json.load(open('../reports/metrics.json'))
rows = []
for arch in ['custom_cnn', 'mobilenetv2', 'efficientnet_b0']:
    h = honest.get(arch, {})
    l = leaky_results.get(arch, {})
    rows.append({
        'model': arch,
        'honest (subject-wise split)': h.get('subject_level_accuracy_softvote'),
        'LEAKY (slice-wise split)': l.get('slice_level_accuracy'),
    })
df = pd.DataFrame(rows).set_index('model')
df['inflation'] = df['LEAKY (slice-wise split)'] - df['honest (subject-wise split)']
display(df.round(3))

ax = df[['honest (subject-wise split)', 'LEAKY (slice-wise split)']].plot.bar(
    figsize=(8, 4.5), rot=0, color=['#3b7dd8', '#d1495b'])
ax.axhline(0.25, ls='--', c='gray', lw=1)
ax.text(-0.4, 0.26, 'chance (4-way)', color='gray', fontsize=9)
ax.set_ylabel('accuracy'); ax.set_ylim(0, 1)
ax.set_title('Same models, same data, one line changed in how the split is made')
plt.tight_layout()

## 6. Where this actually lands

The honest numbers sit in the 50–60% range on a 4-way task, against a 25% chance baseline
and with only 439 subjects. That is roughly what the properly-validated literature reports
for this problem size, and it is the number that would survive contact with a new hospital's
scans.

The leaky numbers are much higher and mean nothing. They are included precisely so the
difference is visible in one figure.

Honest ways to actually move the real number, in rough order of expected payoff:

1. **More subjects.** The 876-scan LONI expansion roughly triples the dataset. Sample size
   is the binding constraint here, not architecture — every architecture tried lands within
   a few points of the others.
2. **Reduce the task.** AD vs CN binary is genuinely easier and genuinely useful; the
   EMCI/LMCI boundary is subtle even for radiologists.
3. **More slices per subject / full 3D.** 2.5D is a cheap step toward this; a real 3D CNN
   is the next one, and needs the larger dataset to be trainable.